## Topic 3: Why Qdrant Specifically

### Concept, Intuition, Why It Exists

- After the previous topic established what the options are, this topic answers the narrower question: given all of those options, why is Qdrant the right choice for this specific project at this specific stage — not as a default answer, but as a reasoned decision against concrete requirements.
- The requirements this project actually has: runs locally without needing a cloud account or a paid tier, persists across restarts when needed, supports metadata filtering during search, is production-ready enough that switching to a hosted version later does not require rewriting client code, and is open-source enough that the full behavior is inspectable rather than a black-box managed service.
- Qdrant meets all five. That is the answer — not "Qdrant is the best vector DB" which is not meaningfully true across all contexts, but "Qdrant is the right fit for these specific requirements at this scale."
- **For this project specifically**: Qdrant's Python client supports an in-memory mode (`QdrantClient(":memory:")`) that needs no Docker, no server, no network download — all the same concepts (collections, points, payloads, filtering, search) work identically. This is what this project uses for learning. The switch to a real persistent server is a one-line change when actually needed.

### Internal Working

- **Rust-based core**: written in Rust, giving memory safety without garbage collection pauses — relevant for latency-sensitive search paths where GC pauses in a Java or Python-based system cause occasional slow outlier queries.
- **HNSW indexing by default**: every collection uses Hierarchical Navigable Small World indexing (covered in full in the next topic). Fast approximate nearest-neighbor search with configurable parameters.
- **Payload filtering during search**: Qdrant applies filters as a constraint inside the HNSW traversal itself — not as a post-search filter applied after retrieving results. This means the k results returned already satisfy the filter. This is architecturally distinct from FAISS, where filtering is always a post-search step.
- **Collections and points**: the primary data model. A collection is a named group of points; each point has an ID, a vector, and a payload (a dict of arbitrary JSON-serializable fields). Covered in full in Topic 5.
- **REST and gRPC**: both APIs expose the same functionality. REST is the default starting point — easier to debug, curl-testable. gRPC is available when throughput matters at scale.
- **Three modes, same client code**:
  - `:memory:` — in-process, no server, no persistence, zero setup. Used in this project for learning.
  - Local Docker — persistent, same codebase as cloud, runs on your machine.
  - Qdrant Cloud — managed hosted version, one-line URL change from local Docker.

### How It's Implemented In This Project

- This project uses `QdrantClient(":memory:")` — no Docker required, no network download, starts instantly inside the Python process.
- All code in this chapter works identically in memory mode and Docker mode. The only thing that changes between them is one line in `get_client()`.
- The collection created (`fd_knowledge_base`) maps directly onto the Document shape from the ingestion chapter: `page_content` becomes text stored in the payload, `metadata` fields become payload fields, and the embedding of `page_content` becomes the point's vector.
- When persistence is actually needed, the upgrade path is: run `docker pull qdrant/qdrant`, start the container, change `QdrantClient(":memory:")` to `QdrantClient(url="http://localhost:6333")`. Nothing else changes.

### Real-World Issues, Edge Cases, Debugging

- **`:memory:` mode loses all data on restart**: every notebook or process restart means re-running the upsert cells. Acceptable for learning, not acceptable for production.
- **Docker volume mount is critical for persistence**: when switching to Docker later, running without `-v` loses all data on container restart. Always use: `docker run -p 6333:6333 -v <path>/qdrant_storage:/qdrant/storage qdrant/qdrant`.
- **Collection already exists error**: if you run the collection creation code twice, Qdrant raises an error — not a warning, an exception. Code must check for existence before creating, or use a `recreate` flag explicitly.
- **Vector dimension mismatch**: a collection is created with a fixed vector size (384 for `paraphrase-multilingual-MiniLM-L12-v2`). Trying to upsert a point with a different-dimensioned vector raises an error immediately. Most common issue during an embedding model swap — covered in Topic 8.
- **Qdrant dashboard**: when running Docker, `http://localhost:6333/dashboard` provides a web UI to inspect collections, run test queries, and verify upserts worked — faster than writing diagnostic print statements during debugging.
- **`:memory:` is not a stripped-down version**: Qdrant's in-memory client is a fully-featured Qdrant instance running in-process — payload filtering, HNSW indexing, all APIs work. It is not a development toy that behaves differently from the production version.

### Design Decisions & Trade-offs

- In-memory for learning, Docker for persistence, Cloud for production — three modes, one codebase, zero rewriting. This is the specific property that makes Qdrant the right choice here: the upgrade path between modes is a one-line change, not a migration.
- Qdrant over Chroma for this project: Chroma's embedded mode also needs no Docker, but does not support payload filtering during search (only post-search), has concurrent-writer limitations in embedded mode, and requires a migration when the project outgrows it. Qdrant's `:memory:` mode gives the same zero-setup experience with full production-equivalent filtering behavior.
- gRPC deferred: REST is used because it is easier to debug at learning stage. gRPC is available and worth knowing exists for production high-throughput scenarios, but premature at this scale.

### Alternatives & When To Use Each

- `QdrantClient(":memory:")` — learning, notebooks, CI tests, any context where persistence is not needed and zero setup matters. What this project uses.
- Local Docker — development and production on your own machine, persistence required, same code as cloud.
- Qdrant Cloud — managed hosting, zero infrastructure, one-line change from local Docker, no vendor lock-in on vector format unlike Pinecone.
- Chroma embedded — if Qdrant's client is not installable for some reason and metadata filtering is not needed. Genuinely simpler to start, but not the right long-term tool.

### Common Mistakes & Production Failures

- Using `:memory:` mode in production and wondering why all vectors disappear on every restart.
- Switching to Docker later without the `-v` volume mount and silently losing all indexed data on the first container restart.
- Creating a collection with one embedding model's vector size, then swapping the embedding model without recreating the collection — Qdrant rejects every upsert with a dimension mismatch error until the collection is rebuilt.
- Not knowing the dashboard exists when running Docker — `http://localhost:6333/dashboard` makes debugging upserts and payloads instant and visual.

### Lead-Level Interview Questions

**Q: Your team cannot get Docker running in a restricted environment. Does that block using Qdrant?**
A: No — `QdrantClient(":memory:")` runs entirely in-process with no server, no Docker, no network. All the same APIs, filtering, and indexing behavior work. The only thing lost is persistence across restarts, which is a real limitation for production but irrelevant for development, testing, and learning.

**Q: What is the actual upgrade path from in-memory Qdrant to a persistent production deployment?**
A: Change one line: `QdrantClient(":memory:")` becomes `QdrantClient(url="http://localhost:6333")` for local Docker, or `QdrantClient(url="https://your-cluster.qdrant.io", api_key="...")` for Qdrant Cloud. Every other line of code — collection creation, upsert, search, filtering — stays identical. This is the specific property that makes `:memory:` mode genuinely useful rather than a throwaway prototype.

**Q: What does payload filtering during search mean in Qdrant, and why does it matter compared to filtering results after the fact?**
A: In Qdrant, a payload filter is applied as a constraint inside the HNSW traversal — the index only considers points matching the filter during traversal, so the k results returned already satisfy the filter. Post-search filtering means fetching a larger result set and discarding non-matching entries afterward, which either wastes the overfetch or returns fewer than k useful results. For the PII and source-scoping use cases in this chapter, filtering during search is the correct semantic.

**Q: Why use Qdrant over Chroma given both can run without Docker?**
A: Chroma's embedded mode removes the server step, which is genuinely convenient. The trade-offs: Chroma does not support payload filtering during search, has concurrent-writer limitations in embedded mode, and requires a migration when the project outgrows it. Qdrant's `:memory:` mode gives the same zero-setup experience with full production-equivalent filtering behavior and a one-line upgrade path to a real server — which is why it is the better long-term choice even at this early stage.

### Hidden Concepts Worth Knowing

- **Collection aliases**: Qdrant supports creating a named alias pointing to a collection. This enables the build-then-swap pattern — ingest into a staging collection, validate, then atomically update the alias to point at the new collection. Live queries always hit the alias, so the swap is instantaneous and no query lands in a half-updated state. Covered in practice in Topic 8.
- **Sparse vectors and hybrid search**: Qdrant supports sparse vectors (BM25-style keyword representations) alongside dense vectors, enabling hybrid search — a combination of semantic similarity and keyword relevance in one query. Relevant when semantic search alone misses exact-match queries like specific FD reference numbers or policy clause numbers.

### Revision Summary

> Qdrant is chosen because it meets all five concrete requirements: zero-setup in-memory mode for learning, Docker for local persistence, Qdrant Cloud for managed hosting — all with identical client code and a one-line upgrade path between modes. `QdrantClient(":memory:")` is what this project uses: no Docker needed, full Qdrant behavior, data lost on restart. When persistence is actually needed, one line changes. Nothing else does.

In [1]:
"""
qdrant_setup.py
-----------------
Qdrant setup and connection verification.

Uses :memory: mode -- no Docker required, no network download,
starts instantly inside the Python process. Full Qdrant behavior
(collections, points, payloads, filtering, search) works identically
in :memory: mode and Docker mode.

The ONLY difference between modes is one line in get_client():
  Learning / no Docker : QdrantClient(":memory:")
  Local Docker         : QdrantClient(url="http://localhost:6333")
  Qdrant Cloud         : QdrantClient(url="https://...", api_key="...")

When you want persistence:
  docker run -p 6333:6333 -v <your_path>/qdrant_storage:/qdrant/storage qdrant/qdrant
  Then swap get_client() to QdrantClient(url="http://localhost:6333").
  Nothing else in this file or any other file needs to change.

Install: pip install qdrant-client
"""

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # paraphrase-multilingual-MiniLM-L12-v2 output dimension


def get_client() -> QdrantClient:
    """Returns a Qdrant client.

    Currently uses :memory: mode -- no Docker needed, zero setup,
    data is lost when the Python process restarts.

    To switch to persistent local Docker, replace with:
        return QdrantClient(url="http://localhost:6333")

    To switch to Qdrant Cloud, replace with:
        return QdrantClient(url="https://your-cluster.qdrant.io", api_key="YOUR_KEY")
    """
    return QdrantClient(":memory:")


def create_collection(client: QdrantClient, recreate: bool = False) -> None:
    """Creates the collection if it does not already exist.

    recreate=True drops and rebuilds the collection -- use this when
    changing embedding models or chunk schemas. Never use on a live
    collection with real data.
    """
    # check what collections already exist so we do not crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    if COLLECTION_NAME in existing:
        if recreate:
            # deliberate teardown -- only when we know we want a clean slate
            client.delete_collection(COLLECTION_NAME)
            print(f"Deleted existing collection '{COLLECTION_NAME}'")
        else:
            print(f"Collection '{COLLECTION_NAME}' already exists -- skipping creation")
            return

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,           # every vector in this collection must be 384 floats
            distance=Distance.COSINE,   # cosine similarity -- right choice for normalized vectors
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}' (vector_size={VECTOR_SIZE})")


def verify_connection(client: QdrantClient) -> None:
    """Quick sanity check -- prints collection info if everything is working."""
    info = client.get_collection(COLLECTION_NAME)
    print(f"Collection   : {COLLECTION_NAME}")
    print(f"Vector size  : {info.config.params.vectors.size}")
    print(f"Points count : {info.points_count}")
    print(f"Status       : {info.status}")


# ----------------------------------------------------------------------
# Run setup
# ----------------------------------------------------------------------

# get a client -- :memory: mode, no Docker needed
client = get_client()

# create the collection
create_collection(client, recreate=False)

# confirm everything is working
verify_connection(client)

Created collection 'fd_knowledge_base' (vector_size=384)
Collection   : fd_knowledge_base
Vector size  : 384
Points count : 0
Status       : green
